# Human DNA Promoter Classification Pipeline
**Goal:** Build a machine learning classifier to identify human promoter using k-mer counting and natural language processing techniques.
**Dataset:** [UCI Molecular Biology Promoter Dataset](https://uci.edu)

In [42]:
# imports
import os
import logging
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

## 1. Data Setup

In [ ]:
# File path
data_path = "data/promoters.data"

# Check if path exists
if not os.path.exists(data_path):
    raise FileNotFoundError(f"File not found")

# Initialize tracking logs
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
# Parse file
records = []
label_map = {'+':1, '-':0}  # translation map for class:label

with open(data_path, 'r') as file:
    for i, line in enumerate(file):
        line = line.strip()  # remove white space from beginning and end of line
        # print(line)
        
        if not line:  # skip empty lines (if \n were not removed properly)
            continue

        # Split line by ',' (each line should have 3 parts only)
        # Only allow max split of 2 to isolate Class and ID; rest of the line will be for sequence if there are extra commas
        parts = line.split(',', 2)

        # Check for indexing errors if line is too short 
        if len(parts) < 3:
            logging.warning(f"SDkipping malformed row on line {i}: Insufficient delimeters.")
            continue

        csym = parts[0].strip()
        name = parts[1].strip()
        seq = parts[2].replace(',', '').strip().lower()  # incase the sequence contains commas within it
        label = label_map.get(csym, -1)

        if label == -1: 
            logging.warning(f"Unrecognized class symbol '{csym}' on line {i}")

        records.append({'Class': csym, 'ID': name, 'Sequence': seq, 'Label': label})

df = pd.DataFrame(records)
df.head()


,Class,ID,Sequence,Label
0,+,S10,tactagcaatacgcttgcgttcggtggttaagtatgtataatgcgc...,1
1,+,AMPC,tgctatcctgacagttgtcacgctgattggtgtcgttacaatctaa...,1
2,+,AROH,gtactagagaactagtgcattagcttatttttttgttatcatgcta...,1
3,+,DEOP2,aattgtgatgtgtatcgaagtgtgttgcggagtagatgttagaata...,1
4,+,LEU1_TRNA,tcgataattaactattgacgaaaagctgaaaaccactagaatgcgc...,1


## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Class Distribution
# .value_counts() - returns Series containing counts of unique values
class_distribution = df['Class'].value_counts(normalize=True)*100
logging.info(f"Class Balance Ratio:\n{class_distribution.to_string()}\n")

# Sequence Length Profile
df['Seq_Length'] = df['Sequence'].apply(len)  # find the length of each sequence 
mean_len = df['Seq_Length'].mean()
max_len = df['Seq_Length'].max()
min_len = df['Seq_Length'].min()
logging.info(f"Sequence Dimensions - Min: {min_len}bp, Max: {max_len}bp, Mean: {mean_len}bp\n")
df.drop(columns=['Seq_Length'], inplace=True)

# Calculate GC Content Percentage (Promoter regions are GC-rich)
def calculate_gc_content(seq: str) -> float:
    """
    Calculate GC Content Percentage of a raw DNA sequence string.

    BIOLOGICAL SIGNFICANCE: 
    Promoter regions are structurally 'GC-rich' compared to the rest of the genome.
    High GC concentration stabilizes transcription factor binding sites, making it a critical feature 
    """
    if not seq:
        return 0.0
    gc_count = seq.count('g') + seq.count('c')
    gc_percentage = (gc_count/len(seq))*100
    return gc_percentage
df['GC_Content'] = df['Sequence'].apply(calculate_gc_content)
mean_gc = df['GC_Content'].mean()  # establishes biological baseline
std_gc = df['GC_Content'].std()  # measure data distribution; high = sequences have diverse compositional variety
logging.info(f"Biological Integrity Summary:\n - Baseline Mean GC Content: {mean_gc:.2f}%\n - Standard Deviation of GC Variant: {std_gc:.2f}%")
     

INFO: Class Balance Ratio:
Class
+    50.0
-    50.0

INFO: Sequence Dimensions - Min: 57bp, Max: 57bp, Mean: 57.0bp

INFO: Biological Integrity Summary:
 - Baseline Mean GC Content: 45.60%
 - Standard Deviation of GC Variant: 7.96%


## 3. Feature Extraction (Genomic Tokenization)

### Methodology: 6-Mer Sliding Window
Machine learning models cannot compute raw, continuous strings of text directly. To resolve this, we implement a sliding-window tokenization technique to break our 57-base-pair DNA sequences into overlapping words of length $k=6$ (6-mers). This effectively translates a unified biological string into a natural language "sentence" of genomic tokens.

### Why Tokenization & 6-Mers are Professionally Necessary:
1. **Contextual Feature Engineering**: In genomics, individual base pairs ($A, C, G, T$) carry no statistical weight on their own. The functional signal depends entirely on multi-base combinations (motifs). For example, transcription factors lock onto specific landing pads that are typically 6–12 base pairs long. A 6-mer window preserves these critical local combinations.
2. **Optimizing Vocabulary Specificity**: DNA has a small 4-letter alphabet. A 3-mer window creates a vocabulary of only $4^3 = 64$ possible words, meaning almost every word appears uniformly across both promoters and background junk DNA, creating massive noise. A 6-mer window expands our high-dimensional vocabulary to $4^6 = 4,096$ unique combinations, allowing the downstream vectorizer to detect clear statistical signals.


In [31]:
def get_kmers(seq: str, k: int=6) -> str:
    """
    Separate continuous DNA sequence into space-separated strings of overlapping k-mers to leverage text-processing libraries, like scikit-learn's vectorizers without manually managing heavy string-to-index mapping. 

    Args: 
        seq (str): DNA sequence
        k (int): size of k-mer; default 6 to optimize unique combo
    """

    # Check to make sure seq fits kmer/window size
    if len(seq) < k:
        logging.warning(f"Sequence length ({len(seq)}) is shorter than k ({k}).")
        return ""

    # Create a list of k-lenth sequences that appear in the sliding window
    kmers = [seq[i:i+k] for i in range(len(seq)-(k-1))]

    # Return sentence created by kmers
    return " ".join(kmers)

In [ ]:
# Apply tokenization to dataframe sequences
df['kmers'] = df['Sequence'].apply(lambda seq: get_kmers(seq, k=6))

df[['Sequence', 'kmers']].head()

,Sequence,kmers
0,tactagcaatacgcttgcgttcggtggttaagtatgtataatgcgc...,tactag actagc ctagca tagcaa agcaat gcaata caat...
1,tgctatcctgacagttgtcacgctgattggtgtcgttacaatctaa...,tgctat gctatc ctatcc tatcct atcctg tcctga cctg...
2,gtactagagaactagtgcattagcttatttttttgttatcatgcta...,gtacta tactag actaga ctagag tagaga agagaa gaga...
3,aattgtgatgtgtatcgaagtgtgttgcggagtagatgttagaata...,aattgt attgtg ttgtga tgtgat gtgatg tgatgt gatg...
4,tcgataattaactattgacgaaaagctgaaaaccactagaatgcgc...,tcgata cgataa gataat ataatt taatta aattaa atta...


## 4. Pipeline Vectorization & Model Training

### Methodology: Bag-of-Words & Multinomial Naive Bayes
In this phase, we map our text tokens into a numerical frequency matrix using a **Bag-of-Words** approach via `CountVectorizer`, which we will then pass into a **Multinomial Naive Bayes Classifier**.

### Why Vectorization is Bundled with Training:
1. **Preventing Data Leakage**: Vectorization is technically a feature extraction step. However, it must be fit exclusively on our training data split. If we vectorize the entire dataset before splitting, the vectorizer will "learn" vocabulary frequencies from the test set, creating an enterprise code bug known as data leakage. This causes artificially inflated validation scores that fail in production.
2. **Defensive Modeling**: We split our raw 6-mer text sentences *first*, fit the vectorizer solely on the training tokens to establish our vocabulary, and then transform the unseen validation tokens against that fixed dictionary. This ensures our evaluation remains unbiased and reflects real-world performance.


In [38]:
# Seperate feature ('k-mer') with labels ('Label')
X = df['kmers'].values
y = df['Label']

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Production Extraction: 
# Fit vectorizer on training tokens
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train)
# Transform test tokens aigainst learned dictionary to prevent leakage
X_test = vectorizer.transform(X_test)
logging.info(f"Train matrix shape: {X_train.shape} | Test matrix shape: {X_test.shape}")

# Train Naive Bayes Classifier
clf = MultinomialNB(alpha=0.1)
clf.fit(X_train, y_train)
logging.info("Model training complete.")

# Evaluate predictions on unseen data
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Validation Accuracy: {accuracy * 100:.2f}%\n")
print("Detailed Classification Performance:")
print(classification_report(y_test, y_pred, target_names=['Non-Promoter', 'Promoter']))

INFO: Train matrix shape: (84, 2117) | Test matrix shape: (22, 2117)
INFO: Model training complete.


Validation Accuracy: 95.45%

Detailed Classification Performance:
              precision    recall  f1-score   support

Non-Promoter       1.00      0.91      0.95        11
    Promoter       0.92      1.00      0.96        11

    accuracy                           0.95        22
   macro avg       0.96      0.95      0.95        22
weighted avg       0.96      0.95      0.95        22



## 5. Production Model Inference
Demonstrate utility of pipeline by feeding two _synthetic control sequences_ into trained model.
* **Sequence 1**: mimic known promoter characteristics (highly stable, GC-rich motifs)
* **Sequence 2**: random generation representing non-promoter background genomic noise


In [44]:
# unseen DNA sequences 
# Sequence 1: mimics known promoter characterstics
# Sequence 2: random background noise DNA
new_seqs = [
    "gctgctaacgcatctttgatagtatgtgttgtaactagaataccataagcttgtttgca", # Expected: Promoter (1)
    "gatcctcccgaataccagcacatctgtcaagcctcaatcctatgaatggatcaaggata"  # Expected: Non-Promoter (0)
]

pmi_df = pd.DataFrame({'Sequence': new_seqs})

# Step 3: Tokenize sequences using get_kmers
pmi_df['kmers'] = pmi_df['Sequence'].apply(lambda seq: get_kmers(seq, k=6))

# Step 4: Vectorize 
X_new = vectorizer.transform(pmi_df['kmers'].values)  # transform matrix version not pd version

# Generate Predictions using trained model
predictions = clf.predict(X_new)
probs = clf.predict_proba(X_new)  # calc confidence level

# Readout
class_labels = {1: "+", 0: "-"}
print("Production Inference Results:\n")
for i, seq in enumerate(new_seqs):
    label = class_labels[predictions[i]]
    confidence = probs[i][predictions[i]]*100 
    print(f"Sample {i+1}: {seq[:20]}")
    print(f"Predicted Class: {label}")
    print(f"Model Confidence: {confidence:.2f}%\n")


Production Inference Results:

Sample 1: gctgctaacgcatctttgat
Predicted Class: +
Model Confidence: 100.00%

Sample 2: gatcctcccgaataccagca
Predicted Class: -
Model Confidence: 100.00%



##